In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

base = expfolder( "system" )[2]

runsim = true

In [ ]:
include( "size-0-params.jl" )
ρlist = ρlist[7:7];  Nρ = length( ρlist )
slist = slist[Ns:Ns];  Ns = length( slist )

# System parameters.
N = 128

# Extract critical noise.
folderdata = [findfolder( N, ρlist[i], slist[j]; base=base, crit=true ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import area and perimeter scaling values.
βdata = first.( readdlm.( folderdata.*"area-perimeter-scaling.txt" ), 2 )

# Import the cluster change frequency values.
θdatalist = readdlm.( folderdata.*"optimal-rates-hat.txt" )

# Extract the power law.
λdata = first.( readdlm.( folderdata.*"power-law.txt" ) )

# Unpack the kernel parameters.
γdata = first.( readdlm.( folderdata.*"kernel-parameters-1.txt" ) );

In [ ]:
# Unpack the probability map data.
ℓdata = [polyloglaw.( 1:N; β=βdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns]
Adata = [(ℓdata[i,j].^2)./(4π) for i ∈ 1:Nρ, j ∈ 1:Ns]

# Construct the probability map.
Ωdata = [[
    P( k2; N=N, i=k1, λℓ=θdatalist[i,j][k1,1], λr=θdatalist[i,j][k1,2], κ=θdatalist[i,j][k1,3] )
    for k1 in 1:N, k2 in 1:N] for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Import cluster mass distribution extracted from data.
Fdata = vec.( readdlm.( folderdata.*"cluster-frequency.txt" ) )
fdata = [Fdata[i,j]./sum( Fdata[i,j] ) for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Simulate the master equations in the larger system size..
T = 1.0
f̂data = [runsim ? kernelloss(
    round.( defInt, Fdata[i,j] ), [γdata[i,j]], (N, Ωdata[i,j], Adata[i,j], ℓdata[i,j]); T=T, verbose=true )[2] :
    vec( readdlm( folderdata[i,j]*"cluster-distribution-hat.txt" ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Extract the power-law of the extrapolated cluster mass distribution.
λ̂data = [
    10^optimize( createzipfloss( N, N.*f̂data[i,j] ), [log10( 1.5 )] ).minimizer[1]
    for i ∈ 1:Nρ, j ∈ 1:Ns]

# Compute the absolute difference between the distribution exponents.
δλdata = abs.( λdata .- λ̂data )./λdata;
δλmax = 0.01round( maximum( δλdata )/0.01 );

In [ ]:
# Define local kernels.
# i = 1;  j = 1
# coll, frag = createkernels( N, Adata[i,j], ℓdata[i,j], [γdata[i,j]] )
# params = Params( N, Ωdata[i,j], coll, frag )

# Test the log-space simulator.
# results, f̂, f = logsimulate( params; f=zipf.( 1:N; b=N, λ=λdata[i,j] ), T=T, verbose=true )

In [ ]:
i = 1;  j = Ns
println( "Density: ", ρlist[i] )
println( "Speed:   ", slist[j] )

# Plot bounds.
ιlist = 0.0 .< Fdata[i,j]
xmax = ceil( log10( N ) )
ymin = floor( minimum( log10.( fdata[i,j][ιlist] ) ) )

plt = plot(; layout=@layout( [a{0.6w} [b; c{0.90h}]] ), size=(400,200), legend=false )

# Plot the final distribution from the mean-field equations against the measured.
plot!( plt[1]; legend=:bottomleft, right_margin=0pt )

scatter!( plt[1], fdata[i,j][ιlist]; color=:gray, markerstrokewidth=0, label=" raw data" )
scatter!( plt[1], f̂data[i,j][ιlist]; color=:black, markersize=3, markerstrokewidth=0, lw=2, label=" master eq." )

xticks = round.( defInt, 2.0.^(0:2:8) )
plot!( plt[1]; xlims=extrema( xticks ), xscale=:log10, xticks=(xticks,string.(xticks)) )
plot!( plt[1]; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt[1]; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)" )
plot!( plt[1]; title="(a)", titleposition=:left )

# Plot the colorbar.
plot!( plt[2]; bottom_margin=5pt, right_margin=15pt )

xmap = 0:0.001:δλmax
ymap = [0,1]
zmap = hcat( xmap, xmap )
heatmap!( plt[2], xmap, ymap, zmap'; color=cgrad( :ice ) )

plot!( plt[2]; xlims=(0,δλmax+eps()), xticks=[0,δλmax] )
plot!( plt[2]; yticks=[] )
annotate!( plt[2], δλmax/2, -4.5, text( L"\frac{|λ - \hat{λ}\ |}{λ}", "Computer Modern", 8, :center ) )
plot!( plt[2]; title="(b)", titleposition=:left )

# Plot the slope of the power-law.
heatmap!( plt[3], ρlist, slist, δλdata; color=cgrad( :ice ) )

plot!( plt[3]; xlims=extrema( ρlist ), xscale=:log10 )
plot!( plt[3]; ylims=extrema( slist ), yscale=:log10 )
plot!( plt[3]; xlabel="density, "*L"ρ", ylabel="speed, "*L"s" )

plot!( plt; framestyle=:box )

# saveplot( plt, figurefolder*"master-equations-cmd_N-$(N)"; background=:white, dpi=600 );

In [ ]:
if false & runsim
    # Save the extrapolated distribution.
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( folderdata[i,j]*"cluster-distribution-hat.txt", f̂data[i,j] )
    end
end